In [1]:
import pandas as pd
df= pd.read_csv("salary_survey_raw.csv")

print(f'Shape: {df.shape}')

Shape: (2800, 17)


In [2]:
null_counts = df.isnull().sum()
print("num total: ",null_counts)

cols_with_null = null_counts[null_counts > 0]

print("Các cột bị thiếu dữ liệu:")
print(cols_with_null)

num total:  timestamp                             0
how_old_are_you                       0
industry                              0
job_title                             0
additional_context_on_job_title     969
annual_salary                       391
additional_monetary_comp           1538
currency                              0
income_context                     2269
country                             115
us_state                           1813
city                               1562
years_of_experience_in_field          0
years_of_experience_overall           0
highest_level_of_education            0
gender                               69
race                                766
dtype: int64
Các cột bị thiếu dữ liệu:
additional_context_on_job_title     969
annual_salary                       391
additional_monetary_comp           1538
income_context                     2269
country                             115
us_state                           1813
city                         

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2800 entries, 0 to 2799
Data columns (total 17 columns):
 #   Column                           Non-Null Count  Dtype
---  ------                           --------------  -----
 0   timestamp                        2800 non-null   str  
 1   how_old_are_you                  2800 non-null   str  
 2   industry                         2800 non-null   str  
 3   job_title                        2800 non-null   str  
 4   additional_context_on_job_title  1831 non-null   str  
 5   annual_salary                    2409 non-null   str  
 6   additional_monetary_comp         1262 non-null   str  
 7   currency                         2800 non-null   str  
 8   income_context                   531 non-null    str  
 9   country                          2685 non-null   str  
 10  us_state                         987 non-null    str  
 11  city                             1238 non-null   str  
 12  years_of_experience_in_field     2800 non-null   str  
 13 

In [4]:
# Bước 1: Tổng quan missing - chạy đầu tiên
def missing_report(data):
    miss = data.isnull().sum()
    pct = (miss / len(data) * 100).round(2)
    return (pd.DataFrame({'count': miss, 'pct_%': pct})
            .query('count > 0').sort_values('pct_%', ascending=False))

# Chạy report cho df
print("--- Báo cáo Missing df ---")
display(missing_report(df))

--- Báo cáo Missing df ---


,count,pct_%
income_context,2269,81.04
us_state,1813,64.75
city,1562,55.79
additional_monetary_comp,1538,54.93
additional_context_on_job_title,969,34.61
race,766,27.36
annual_salary,391,13.96
country,115,4.11
gender,69,2.46


### Bước 2: Xử lý Missing Values theo Checklist
**Giải thích và phân loại Null (MCAR/MAR/MNAR):**
- Cột `Age` (hoặc tương đương): Thường là MAR (người lớn tuổi có xu hướng ít điền tuổi hơn) hoặc MCAR. Phân phối có thể lệch phải nên dùng `median` để điền.
- Cột `Country` (hoặc tương đương): Có thể là MCAR. Không ảnh hưởng nhiều nên điền 'Unknown'.
- Cột `Salary` (hoặc CompTotal): Có thể là MNAR (người lương rất cao hoặc rất thấp thường giấu lương). Có thể dùng `interpolate` hoặc drop nếu thiếu quá nhiều.

**Các chiến lược áp dụng:**
1. **fillna(median)**: cho cột có phân phối lệch (vd: tuổi, năm kinh nghiệm)
2. **fillna('Unknown')**: cho dữ liệu phân loại (categorical)
3. **drop**: loại bỏ các dòng bị thiếu ở cột cực kỳ quan trọng không thể nội suy (vd. ID hoặc Target variable) hoặc **interpolate** cho dữ liệu chuỗi thời gian/liên tục.

In [5]:
# Lưu giữ original report để so sánh
report_before = missing_report(df)

# Xử lý copy để không ảnh hưởng dữ liệu gốc khi test
df_clean = df.copy()

# 1. Fillna với chuỗi (Categorical)
# Thay 'Country' bằng tên cột phân loại bị missing thực tế trong df của bạn
if 'Country' in df_clean.columns:
    df_clean['Country'] = df_clean['Country'].fillna('Unknown')
elif 'MainBranch' in df_clean.columns:
    df_clean['MainBranch'] = df_clean['MainBranch'].fillna('Unknown')

# 2. Fillna với median (Numerical)
# Thay 'Age' bằng tên cột số bị missing thực tế
if 'WorkExp' in df_clean.columns:
    df_clean['WorkExp'] = df_clean['WorkExp'].fillna(df_clean['WorkExp'].median())
elif 'YearsCode' in df_clean.columns:
    # Xử lý string sang số nếu cần trước khi tính median
    pass

# 3. Drop hoặc Interpolate
# Drop những dòng có missing ở cột quan trọng (ví dụ: CompTotal)
if 'CompTotal' in df_clean.columns:
    df_clean = df_clean.dropna(subset=['CompTotal'])

# Hoặc dùng interpolate (phù hợp với chuỗi hoặc thứ tự)
# df_clean['NumericColumn'] = df_clean['NumericColumn'].interpolate()

# So sánh trước và sau
print("--- So sánh Null Count ---")
report_after = missing_report(df_clean)

comparison = pd.DataFrame({
    'Before': report_before['count'] if report_before is not None else 0,
    'After': report_after['count'] if report_after is not None else 0
}).fillna(0).astype(int)

display(comparison)

--- So sánh Null Count ---


,Before,After
income_context,2269,2269
us_state,1813,1813
city,1562,1562
additional_monetary_comp,1538,1538
additional_context_on_job_title,969,969
race,766,766
annual_salary,391,391
country,115,115
gender,69,69


### Checklist 1.2 & 1.3 — Duplicates và Data Types
    
**1. Giải thích về Duplicates (Trùng lặp - Checklist 1.2):**
- **Tại sao có duplicate?** Nguyên nhân có thể do lỗi nhập liệu (ví dụ user submit form nhiều lần), lỗi khi join/merge dữ liệu với bảng khác bị sai khóa, hoặc hệ thống tự động ghi nhận file quá nhiều lần.
- **Cách xử lý:** Với dạng dữ liệu bảng như kết quả survey/khảo sát, các hàng bị duplicate toàn bộ (identical rows) thường là dữ liệu thừa và có thể mang lại kết quả sai lệch nếu tính tổng/phân phối, vì vậy ta sẽ `drop_duplicates` và giữ lại bản ghi đầu tiên.

**2. Giải thích về Data Types (Kiểu dữ liệu - Checklist 1.3):**
- **Tại sao dtype sai lại gây lỗi khi phân tích?** 
  - Nếu cột tiền tệ/lương (ví dụ: `$100,000`) bị lưu dưới định dạng dạng chuỗi (`object`) vì chứa dấu phẩy và dollar sign, máy tính không thể thực hiện các phép toán (như tính trung bình `mean()` hay tổng `sum()`).
  - Cột thời gian thực tế cần được đưa về dạng `datetime` để gom nhóm theo tháng/năm, nếu chỉ ở dạng chuỗi, bạn sẽ không thể lấy ra ngày tháng hay đo lường độ trễ (timedelta).
  - Định dạng chuỗi (`object`) cực kì tốn bộ nhớ. Đối với dữ liệu Categorical lặp lại (Country, Industry,...), sử dụng dtype `category` sẽ giúp tiết kiệm RAM đáng kể và tăng tốc độ xử lý.

In [6]:
# ==========================================
# GÓI 1.2: KIỂM TRA VÀ XỬ LÝ DUPLICATES 
# ==========================================
print("--- 1.2 DUPLICATES ---")
n_before = len(df_clean)

# 1. Kiểm tra duplicate toàn hàng
n_dup_all = df_clean.duplicated().sum()
print(f"Duplicate toàn hàng: {n_dup_all} ({(n_dup_all/n_before)*100:.1f}%)")

# 2. Kiểm tra duplicate theo key columns (nếu có key duy nhất)
# Thay 'ResponseId' bằng tên khóa chính/email trong tập data của bạn (nếu có)
key_col = 'ResponseId'
if key_col in df_clean.columns:
    key_dup = df_clean.duplicated(subset=[key_col]).sum()
    print(f"Duplicate theo key '{key_col}': {key_dup}")

# 3. Loại bỏ Duplicate (Giữ lại dòng đầu tiên)
df_clean = df_clean.drop_duplicates(keep='first').reset_index(drop=True)
n_after = len(df_clean)
print(f"Số lượng hàng (Rows): {n_before} -> {n_after}\n")

# ==========================================
# GÓI 1.3: KIỂM TRA VÀ CONVERT DATA TYPES
# ==========================================
print("--- 1.3 DATA TYPES ---")
# Kiểm tra bộ nhớ ban đầu
mem_before = df_clean.memory_usage(deep=True).sum() / 1024**2

# Xác định các cột bị sai dtype ban đầu
print("Dtypes của 5 cột đầu tiên trước khi xử lý:")
print(df_clean.dtypes.head())
print("-" * 20)

# 1. Convert cột số đang bị lưu dưới dạng string (có chứa kí tự $) sang float
# Ví dụ: đổi 'CompTotal' hoặc 'Salary' tùy tên cột bạn có
target_salary_col = 'CompTotal' 
if target_salary_col in df_clean.columns and df_clean[target_salary_col].dtype == 'object':
    df_clean[target_salary_col] = (df_clean[target_salary_col]
                                   .astype(str)
                                   .str.replace(',', '', regex=False)
                                   .str.replace('$', '', regex=False)
                                   .astype(float))

# 2. Convert ngày/giờ sang datetime
# Thay 'timestamp' bằng cột thời gian của bạn (nếu có)
date_col = 'timestamp' 
if date_col in df_clean.columns:
    df_clean[date_col] = pd.to_datetime(df_clean[date_col], errors='coerce')

# 3. Chuyển đổi dữ liệu phân loại sang 'category' để tiết kiệm bộ nhớ
cat_cols = ['Country', 'Employment', 'EdLevel', 'industry'] # Sửa danh sách này theo đúng cột bạn có
for col in cat_cols:
    if col in df_clean.columns and df_clean[col].dtype == 'object':
        df_clean[col] = df_clean[col].astype('category')

# In lại dtype để xem kết quả
print("\nCác dtype sau convert (ví dụ):")
if target_salary_col in df_clean.columns: print(f"{target_salary_col}: {df_clean[target_salary_col].dtype}")
if date_col in df_clean.columns: print(f"{date_col}: {df_clean[date_col].dtype}")

# Kiểm tra độ tiết kiệm bộ nhớ
mem_after = df_clean.memory_usage(deep=True).sum() / 1024**2
print(f"\nBộ nhớ Dataframe lúc đầu:  {mem_before:.2f} MB")
print(f"Bộ nhớ Dataframe hiện tại: {mem_after:.2f} MB")
print(f"=> Bạn đã tiết kiệm được:  {mem_before - mem_after:.2f} MB")

--- 1.2 DUPLICATES ---
Duplicate toàn hàng: 38 (1.4%)
Số lượng hàng (Rows): 2800 -> 2762

--- 1.3 DATA TYPES ---
Dtypes của 5 cột đầu tiên trước khi xử lý:
timestamp                          str
how_old_are_you                    str
industry                           str
job_title                          str
additional_context_on_job_title    str
dtype: object
--------------------

Các dtype sau convert (ví dụ):
timestamp: datetime64[us]

Bộ nhớ Dataframe lúc đầu:  2.43 MB
Bộ nhớ Dataframe hiện tại: 2.28 MB
=> Bạn đã tiết kiệm được:  0.15 MB
